In [1]:
%clear
import pandas as pd

## Calculate means

In [4]:
SAMPLE_IDS_FILE = r"C:\Users\papan\Desktop\Coffeeomics\sample_information.csv"
MEATABOLOMICS_FILE = r"C:\Users\papan\Desktop\Coffeeomics\metabolomics\data\metabolomics_counts.csv"
METABOLOMICS_RESULTS_FOLDER = r"C:\Users\papan\Desktop\Coffeeomics\metabolomics\mann_whitney_results"
P_VALUE_THRESHOLD = 0.05
CONDITIONS = {
    "species": ["C. arabica", "C. canephora"],
    "temperature": ("25/20°C", "37/28°C", "42/30°C", "Rec 14 - 25/20°C"),
    "co2": ("380ppm", "700ppm"),
    "drought": ("without", "medium", "severe")
}

In [5]:
samples = pd.read_csv(SAMPLE_IDS_FILE, index_col=0)
samples = samples.rename(columns={
    "Species": "species",
    "Temperature": "temperature",
    "CO2_ppm": "co2",
    "Water_Stress": "drought",
})
samples["co2"] = samples["co2"].astype(str)

# sample_information.csv lists samples across ALL omics layers; keep only
# those actually profiled for metabolomics, or downstream lookups (e.g. '7A',
# which has Metabolomics=False) raise a KeyError against metabolomics_counts.csv
samples = samples[samples["Metabolomics"]]

In [6]:
# metabolomics_counts.csv has a 5-row header (Species, Temperature, CO2, Water_Stress,
# Sample_ID) -- must be read as a MultiIndex header, then flattened to the Sample_ID
# level only. Reading with a plain header=0 (as before) silently collapses the file:
# every column becomes a duplicated species name and the real header rows leak into
# the data as spurious index entries.
metabolomics = pd.read_csv(MEATABOLOMICS_FILE, index_col=0, header=[0, 1, 2, 3, 4])
metabolomics.columns = metabolomics.columns.get_level_values(-1)  # keep only Sample_ID
metabolites = metabolomics.index.values.tolist()

# Reconcile against the actual data columns -- sample_information.csv's Metabolomics
# flag and metabolomics_counts.csv's columns can disagree (e.g. a sample flagged True
# but later dropped in QC, or added to the data without the metadata being updated).
only_in_metadata = set(samples.index) - set(metabolomics.columns)
only_in_data     = set(metabolomics.columns) - set(samples.index)
if only_in_metadata:
    print(f"Warning: {len(only_in_metadata)} sample(s) flagged Metabolomics=True in "
          f"sample_information.csv but missing from metabolomics_counts.csv: "
          f"{sorted(only_in_metadata)}")
if only_in_data:
    print(f"Warning: {len(only_in_data)} sample(s) present in metabolomics_counts.csv "
          f"but not flagged Metabolomics=True (or absent) in sample_information.csv: "
          f"{sorted(only_in_data)}")
samples = samples[samples.index.isin(metabolomics.columns)]

In [7]:
mean_df = pd.DataFrame(index=metabolomics.index.values.tolist()).rename_axis("metabolites")
for species in CONDITIONS["species"]:
    for temperature in CONDITIONS["temperature"]:
        for co2 in CONDITIONS["co2"]:
            for drought in CONDITIONS["drought"]:
                sample_ids = samples[(samples["species"] == species) & (samples["temperature"] == temperature) & (samples["co2"] == co2) & (samples["drought"] == drought)].index.values.tolist()
                if sample_ids:      # Only proceeds if data for the set of conditions exists
                    means = ((metabolomics[sample_ids].mean(axis=1).to_frame())
                             .rename(columns={0: f"{species}_{temperature}_{co2}_{drought}"}))\
                             .rename_axis("metabolites")
                    mean_df = mean_df.merge(means, on="metabolites")
mean_df.to_csv(METABOLOMICS_RESULTS_FOLDER + "/metabolomics_means.csv")



## Mann-Whitney U Test

In [8]:
from scipy.stats import mannwhitneyu

In [9]:
temperature_comparisons = {
    "25/20°C": ["37/28°C", "42/30°C", "Rec 14 - 25/20°C"],
    "37/28°C": ["42/30°C", "Rec 14 - 25/20°C"],
    "42/30°C": ["Rec 14 - 25/20°C"]
}
# CO2 not needed as only 380ppm vs 700ppm can be compared
drought_comparisons = {
    "without": ["medium", "severe"],
    "medium": ["severe"]
}

In [10]:
def run_mannwhitneyu(a_sample_ids: list[str], b_sample_ids: list[str]) -> pd.DataFrame:
    a = metabolomics[a_sample_ids]
    b = metabolomics[b_sample_ids]
    result_df = a.apply(
        lambda row: mannwhitneyu(
            row.dropna(),
            b.loc[row.name].dropna(),
            alternative="two-sided"
        ),
        axis=1,
        result_type="expand"
    )
    result_df.columns = ["u_statistic", "p_value"]
    return result_df


In [11]:
def export_mannwhiteneyu(dfs: list[pd.DataFrame], stressor: str) -> None:
    species = CONDITIONS["species"]
    with pd.ExcelWriter(
        f"{METABOLOMICS_RESULTS_FOLDER}/mannwhiteneyu_{stressor}.xlsx",
        engine="openpyxl"
    ) as writer:
        for df, sp in zip(dfs, species):
            # Only highlight p_value columns -- df.style.map() applies to every cell,
            # so without this subset it was also (meaninglessly) highlighting any
            # u_statistic that happened to fall between 0 and P_VALUE_THRESHOLD.
            p_value_cols = [c for c in df.columns if c[-1] == "p_value"]
            styled = df.style.map(
                lambda v: "background-color: #32a852"
                if P_VALUE_THRESHOLD >= v > 0 else "",
                subset=p_value_cols
            )
            styled.to_excel(writer, sheet_name=sp, merge_cells=True)

In [14]:
temperature_comp_dfs = []
co2_comp_dfs = []
drought_comp_dfs = []

# Temperature comparisons
for species in CONDITIONS["species"]:
    temperature_dfs = []
    temperature_names = []
    for comp_b in temperature_comparisons.keys():
        for comp_a in temperature_comparisons[comp_b]:
            co2_dfs = []
            co2_names = []
            for co2 in CONDITIONS["co2"]:
                drought_dfs = []
                drought_names = []
                for drought in CONDITIONS["drought"]:
                    a_sample_ids = samples[(samples["species"] == species) & (samples["temperature"] == comp_a) & (samples["co2"] == co2) & (samples["drought"] == drought)].index.values.tolist()
                    b_sample_ids = samples[(samples["species"] == species) & (samples["temperature"] == comp_b) & (samples["co2"] == co2) & (samples["drought"] == drought)].index.values.tolist()

                    if a_sample_ids and b_sample_ids:   # Only proceed if both cases exist
                        result_df = run_mannwhitneyu(a_sample_ids, b_sample_ids)
                        drought_dfs.append(result_df)
                        drought_names.append(drought)
                if drought_dfs:
                    drought_combined = pd.concat(drought_dfs, axis=1, keys=drought_names)
                    co2_dfs.append(drought_combined)
                    co2_names.append(co2)
            if co2_dfs:
                co2_combined = pd.concat(co2_dfs, axis=1, keys=co2_names)
                temperature_dfs.append(co2_combined)
                temperature_names.append(comp_b + " vs " + comp_a)
    if temperature_dfs:
        temperature_combined = pd.concat(temperature_dfs, axis=1, keys=temperature_names)
        temperature_comp_dfs.append(temperature_combined)

    # CO2 comparsions
    temperature_dfs = []
    temperature_names = []
    for temperature in CONDITIONS["temperature"]:
        drought_dfs = []
        drought_names = []
        for drought in CONDITIONS["drought"]:
            a_sample_ids = samples[(samples["species"] == species) & (samples["temperature"] == temperature) & (samples["drought"] == drought) & (samples["co2"] == "700ppm")].index.values.tolist()
            b_sample_ids = samples[(samples["species"] == species) & (samples["temperature"] == temperature) & (samples["drought"] == drought) & (samples["co2"] == "380ppm")].index.values.tolist()
            if a_sample_ids and b_sample_ids:   # Only proceed if both cases exist
                result_df = run_mannwhitneyu(a_sample_ids, b_sample_ids)
                drought_dfs.append(result_df)
                drought_names.append(drought)
        if drought_dfs:
            drought_combined = pd.concat(drought_dfs, axis=1, keys=drought_names)
            temperature_dfs.append(drought_combined)
            temperature_names.append(temperature)
    if temperature_dfs:
        temperature_combined = pd.concat(temperature_dfs, axis=1, keys=temperature_names)
        temperature_combined.columns = pd.MultiIndex.from_tuples(
            [("380 vs 700", *col) for col in temperature_combined.columns]
        )
        co2_comp_dfs.append(temperature_combined)

    # Drought comparisons
    drought_dfs = []
    drought_names = []
    for comp_b in drought_comparisons.keys():
        for comp_a in drought_comparisons[comp_b]:
            co2_dfs = []
            co2_names = []
            for co2 in CONDITIONS["co2"]:
                temperature_dfs = []
                temperature_names = []
                for temperature in CONDITIONS["temperature"]:
                    a_sample_ids = samples[(samples["species"] == species) & (samples["co2"] == co2) & (samples["temperature"] == temperature) & (samples["drought"] == comp_a)].index.values.tolist()
                    b_sample_ids = samples[(samples["species"] == species)& (samples["co2"] == co2) & (samples["temperature"] == temperature) & (samples["drought"] == comp_b)].index.values.tolist()

                    if a_sample_ids and b_sample_ids:   # Only proceed if both cases exist
                        result_df = run_mannwhitneyu(a_sample_ids, b_sample_ids)
                        temperature_dfs.append(result_df)
                        temperature_names.append(temperature)
                if temperature_dfs:
                    temperature_combined = pd.concat(temperature_dfs, axis=1, keys=temperature_names)
                    co2_dfs.append(temperature_combined)
                    co2_names.append(co2)
            if co2_dfs:
                co2_combined = pd.concat(co2_dfs, axis=1, keys=co2_names)
                drought_dfs.append(co2_combined)
                drought_names.append(comp_b + " vs " + comp_a)
    if drought_dfs:
        drought_combined = pd.concat(drought_dfs, axis=1, keys=drought_names)
        drought_comp_dfs.append(drought_combined)


export_mannwhiteneyu(temperature_comp_dfs, "temperature")
export_mannwhiteneyu(co2_comp_dfs, "co2")
export_mannwhiteneyu(drought_comp_dfs, "drought")